# E9: Integrated Pipeline — Inspection, Evaluation & Error Analysis

## Three modes

1. **Inspection** (Section 2): Run ONE game step-by-step, print every decision
2. **Evaluation** (Section 3): Run N games, collect metrics
3. **Error Analysis** (Section 4): Post-hoc study of failures

## Pipeline under test

```
RETRIEVE:  causal_search → k episodes with continuations
VALIDATE:  KG.filter_valid_actions → remove invalid actions
SCORE:     BN.query → P(success | task, phase, region, action)
ACT:       LLM chooses from scored + validated actions
LEARN:     outcome → typed_edges + BN.observe + WeightLearner
```

**Model**: Ollama qwen2.5:7b (zero cost)


In [ ]:
import os, json, time, re
import numpy as np
from collections import Counter, defaultdict
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# === CONFIG ===
OLLAMA_URL = 'http://hpc.glaciar.lab:11434/v1'
client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
LLM_MODEL = 'qwen2.5-coder:7b-instruct'

MAX_STEPS = 30
N_EVAL = 30
TOP_K = 3

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print(f'LLM: {LLM_MODEL}')
print(f'Embedder: {embedder.get_sentence_embedding_dimension()}D')


## 1. Build Memory Components

Three components initialized from expert data:
- **CVX Index**: 336 expert episodes with temporal edges
- **Knowledge Graph**: task structure with preconditions/effects  
- **Bayesian Network**: P(success | task_type, phase, action)


In [ ]:
import chronos_vector as cvx

# === CVX Index ===
with open('../data/episodic/alfworld_metadata.json') as f:
    meta = json.load(f)

index = cvx.TemporalIndex(m=16, ef_construction=100)
action_lookup, task_lookup = {}, {}

for ep_id_str, ep in meta.items():
    ep_id = int(ep_id_str)
    task = ep.get('task', '')
    task_lookup[ep_id] = task
    for si, step in enumerate(ep['steps']):
        text = f"{task} | {step.get('action','')} | {step.get('observation','')}"
        vec = embedder.encode(text).tolist()
        ts = ep_id * 10000 + si
        index.insert(ep_id, ts, vec, reward=1.0)
        action_lookup[ts] = step.get('action', '')

print(f'CVX Index: {len(index)} points from {len(meta)} episodes')
print(f'Action lookup: {len(action_lookup)} entries')


In [ ]:
# === Task type detection ===
TASK_TYPES = {
    'heat': 'heat', 'cool': 'cool', 'clean': 'clean',
    'examine': 'examine', 'look at': 'examine',
    'find two': 'pick_two', 'put two': 'pick_two',
}

def detect_task_type(task_text):
    for kw, tt in TASK_TYPES.items():
        if kw in task_text.lower(): return tt
    return 'pick'

STRATEGIES = {
    'pick': 'Find the object, take it, go to target, put it.',
    'heat': 'Find object, take, go to microwave, heat, take, go to target, put.',
    'cool': 'Find object, take, go to fridge, cool, take, go to target, put.',
    'clean': 'Find object, take, go to sinkbasin, clean, go to target, put.',
    'examine': 'Find object, take, find lamp, use lamp.',
    'pick_two': 'Find first, take, put at target. Find second, take, put at target.',
}

# === Phase detection from observation ===
def detect_phase(observation, history):
    """Infer abstract phase from recent action history."""
    if not history:
        return 'searching'
    
    recent = [h['action'].lower() for h in history[-3:]]
    last_action = recent[-1] if recent else ''
    
    # Just transformed (heated/cooled/cleaned) → placing
    if any(a.startswith(('heat', 'cool', 'clean')) and not a.startswith('go') for a in recent):
        if any(a.startswith('take') for a in recent[recent.index(next(a for a in recent if a.startswith(('heat','cool','clean')))):]):
            return 'placing'
    
    # Holding something → check if we need to transform or place
    if any(a.startswith(('take', 'pick')) for a in recent[-2:]):
        return 'holding'
    
    # At an appliance → transforming
    obs_lower = observation.lower()
    if any(x in obs_lower for x in ['microwave', 'fridge', 'sinkbasin']):
        if any(a.startswith(('take', 'pick')) for a in recent[-4:]):
            return 'transforming'
    
    return 'searching'

# === Action type classification ===
def classify_action(action_text):
    a = action_text.lower()
    if a.startswith('go to'): return 'navigate'
    if a.startswith('take') or a.startswith('pick'): return 'take'
    if a.startswith('put'): return 'place'
    if a.startswith('open'): return 'open'
    if a.startswith('close'): return 'close'
    if a.startswith('clean'): return 'clean'
    if a.startswith('heat') or a.startswith('cook'): return 'heat'
    if a.startswith('cool'): return 'cool'
    if a.startswith('use'): return 'use'
    if a.startswith('examine') or a.startswith('look'): return 'examine'
    return 'other'

print('Phase detection + action classification ready')
print(f'Task types: {list(STRATEGIES.keys())}')


In [ ]:
from alfworld.agents.environment.alfred_tw_env import AlfredTWEnv

config = {
    'dataset': {
        'data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/train'),
        'eval_id_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_seen'),
        'eval_ood_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_unseen'),
        'num_train_games': 0, 'num_eval_games': 0,
    },
    'env': {'goal_desc_human_anns_prob': 0.0, 'task_types': [1,2,3,4,5,6],
            'domain_randomization': False, 'expert_type': 'handcoded'},
    'general': {'training_method': 'dqn', 'random_seed': 42},
    'rl': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'dagger': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'logic': {'domain': os.path.expanduser('~/.cache/alfworld/logic/alfred.pddl'),
              'grammar': os.path.expanduser('~/.cache/alfworld/logic/alfred.twl2')},
}
env_wrapper = AlfredTWEnv(config, train_eval='eval_out_of_distribution')
env = env_wrapper.init_env(batch_size=1)
print(f'ALFWorld: {len(env_wrapper.game_files)} games')


## 2. Pipeline Functions

Each step returns diagnostic information for inspection.


In [ ]:
def retrieve_and_align(observation, task, history, verbose=False):
    """Retrieve expert episodes AND find where we are in them.

    Instead of showing the LLM a full plan, we:
    1. Find similar expert episodes
    2. For each, compare our current embedding to EVERY step
    3. The closest step = our probable position
    4. The NEXT step = the recommended action

    Returns: list of (recommended_action, confidence, episode_id, position)
    """
    # Embed current state (same format as indexed steps)
    last_action = history[-1]['action'] if history else 'starting'
    query_text = f'{task} | {last_action} | {observation}'
    query_vec = np.array(embedder.encode(query_text))

    # Step 1: Find relevant episodes via causal_search
    results = index.causal_search(vector=query_vec.tolist(), k=TOP_K * 2, temporal_context=1)

    # Collect unique episode IDs
    seen_eps = set()
    episode_ids = []
    for r in results:
        if r['entity_id'] not in seen_eps:
            seen_eps.add(r['entity_id'])
            episode_ids.append(r['entity_id'])
        if len(episode_ids) >= TOP_K:
            break

    # Step 2: For each episode, align our current state
    recommendations = []
    for ep_id in episode_ids:
        traj = index.trajectory(ep_id)
        if len(traj) < 2:
            continue

        # Get all step embeddings for this episode
        step_vecs = []
        step_actions = []
        for ts, vec in traj:
            step_vecs.append(np.array(vec))
            step_actions.append(action_lookup.get(ts, '?'))

        # Find which step we're closest to (cosine distance)
        distances = []
        for sv in step_vecs:
            cos_sim = np.dot(query_vec, sv) / (np.linalg.norm(query_vec) * np.linalg.norm(sv) + 1e-8)
            distances.append(1.0 - cos_sim)

        best_pos = int(np.argmin(distances))
        best_dist = distances[best_pos]

        # The recommended action = the NEXT step after our position
        if best_pos + 1 < len(step_actions):
            next_action = step_actions[best_pos + 1]
            confidence = 1.0 - best_dist  # higher = more confident

            recommendations.append({
                'action': next_action,
                'confidence': confidence,
                'episode_id': ep_id,
                'position': best_pos,
                'total_steps': len(step_actions),
                'episode_task': task_lookup.get(ep_id, '?'),
                'distance': best_dist,
            })

    # Sort by confidence (highest first)
    recommendations.sort(key=lambda r: -r['confidence'])

    if verbose:
        print(f'  ALIGN: {len(recommendations)} recommendations')
        for i, rec in enumerate(recommendations[:3]):
            print(f'    [{i}] RECOMMEND: "{rec["action"]}" '
                  f'(pos {rec["position"]}/{rec["total_steps"]}, '
                  f'conf={rec["confidence"]:.3f}, ep={rec["episode_id"]})')

    return recommendations

def format_simple_context(recommendations, task_type):
    """Format a MINIMAL context for the LLM.

    Instead of showing full expert plans, show:
    - The strategy (one line)
    - The top recommended action (one line)
    """
    strategy = STRATEGIES.get(task_type, '')
    parts = []
    if strategy:
        parts.append(f'Strategy: {strategy}')

    if recommendations:
        top = recommendations[0]
        parts.append(f'Recommended next action: {top["action"]}')

        # If multiple experts agree, note it
        actions = [r['action'] for r in recommendations[:3]]
        action_types = [classify_action(a) for a in actions]
        if len(set(action_types)) == 1:
            parts.append(f'(all {len(recommendations[:3])} experts agree: {action_types[0]})')

    return '\n'.join(parts) + '\n'

def extract_candidate_actions(results, verbose=False):
    """Extract candidate actions from recommendations."""
    actions = []
    for r in results:
        action_type = classify_action(r['action'])
        actions.append({
            'action': r['action'], 'action_type': action_type,
            'source_episode': r['episode_id'], 'score': r['confidence'],
        })
    if verbose:
        print(f'  EXTRACT: {len(actions)} candidate actions')
        type_counts = Counter(a['action_type'] for a in actions)
        print(f'    Types: {dict(type_counts)}')
    return actions

def llm_decide(observation, admissible, task, history, context, verbose=False):
    """LLM chooses action with minimal context."""
    system = ('You are a household agent. You MUST choose EXACTLY ONE action '
              'from the Available actions list below. Copy the action text '
              'EXACTLY as shown. Do NOT invent actions.')
    user = f'Task: {task}\n\n'
    if context:
        user += context + '\n'
    if history:
        user += 'Recent:\n'
        for h in history[-3:]:  # only last 3, not 5
            user += f'  > {h["action"]} -> {h["obs"][:50]}\n'
        user += '\n'
    user += f'Obs: {observation}\n\nAvailable actions:\n'
    for a in admissible:
        user += f'  - {a}\n'
    user += '\nChoose ONE action:'

    if verbose:
        print(f'  PROMPT ({len(user)} chars):')
        for line in user.split('\n')[:12]:
            print(f'    {line[:80]}')
        if user.count('\n') > 12:
            print(f'    ... ({user.count(chr(10))-12} more lines)')

    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{'role':'system','content':system},{'role':'user','content':user}],
            temperature=0.0, max_tokens=100,
        )
        chosen = resp.choices[0].message.content.strip().lower()
    except:
        chosen = admissible[0].lower()

    for a in admissible:
        if a.lower() == chosen:
            if verbose: print(f'  LLM: exact match "{a}"')
            return a
    for a in admissible:
        if a.lower() in chosen or chosen in a.lower():
            if verbose: print(f'  LLM: partial match "{a}"')
            return a
    if verbose: print(f'  LLM: fallback (raw: "{chosen[:40]}")')
    return admissible[0]

print('Pipeline functions ready (temporal alignment + minimal context)')


## 2. Inspection Mode

Run ONE game step-by-step. See every decision the pipeline makes.


In [ ]:
def run_game_inspect(env):
    """Run one game with full diagnostic output."""
    obs, info = env.reset()
    observation = obs[0]
    task = observation.split('Your task is to:')[-1].strip().split('\n')[0] if 'Your task is to:' in observation else ''
    task_type = detect_task_type(task)

    print(f'═══ GAME: {task} ═══')
    print(f'Task type: {task_type}')
    print(f'Strategy: {STRATEGIES.get(task_type, "?")}')
    print()

    history = []
    step_log = []

    for step in range(MAX_STEPS):
        admissible = info['admissible_commands'][0]
        phase = detect_phase(observation, history)

        print(f'── Step {step+1} ── phase={phase}')
        print(f'  OBS: {observation[:100]}')
        print(f'  ADMISSIBLE ({len(admissible)}): {admissible[:5]}...')

        # 1. RETRIEVE + ALIGN
        recommendations = retrieve_and_align(observation, task, history, verbose=True)

        # 2. FORMAT minimal context
        context = format_simple_context(recommendations, task_type)

        # 3. LLM DECIDE
        action = llm_decide(observation, admissible, task, history, context, verbose=True)

        step_log.append({
            'step': step, 'phase': phase, 'action': action,
            'recommended': recommendations[0]['action'] if recommendations else None,
            'obs': observation[:100],
        })

        # Execute
        obs, rewards, dones, info = env.step([action])
        observation = obs[0]
        history.append({'action': action, 'obs': observation[:100]})

        print(f'  RESULT: {observation[:80]}')
        print()

        if dones[0]: break

    won = info['won'][0]
    print(f'{"WIN" if won else "FAIL"} in {len(history)} steps')
    return {'task': task, 'task_type': task_type, 'won': won,
            'steps': len(history), 'step_log': step_log, 'history': history}

result = run_game_inspect(env)


## 3. Evaluation Mode

Run N games, collect aggregate metrics.


In [ ]:
def run_game_eval(env):
    """Run one game silently, return result."""
    obs, info = env.reset()
    observation = obs[0]
    task = observation.split('Your task is to:')[-1].strip().split('\n')[0] if 'Your task is to:' in observation else ''
    task_type = detect_task_type(task)
    history = []
    phases_seen = []
    action_types_used = []
    aligned_count = 0

    for step in range(MAX_STEPS):
        admissible = info['admissible_commands'][0]
        phase = detect_phase(observation, history)
        phases_seen.append(phase)

        recommendations = retrieve_and_align(observation, task, history)
        context = format_simple_context(recommendations, task_type)
        action = llm_decide(observation, admissible, task, history, context)

        # Track if LLM followed the recommendation
        if recommendations and action.lower() == recommendations[0]['action'].lower():
            aligned_count += 1

        action_types_used.append(classify_action(action))
        obs, rewards, dones, info = env.step([action])
        observation = obs[0]
        history.append({'action': action, 'obs': observation[:100]})
        if dones[0]: break

    won = info['won'][0]
    return {
        'task': task, 'task_type': task_type, 'won': won,
        'steps': len(history), 'history': history,
        'phases': phases_seen, 'action_types': action_types_used,
        'aligned': aligned_count,
    }

print(f'Running {N_EVAL} games...')
all_results = []
wins = 0
for g in range(N_EVAL):
    r = run_game_eval(env)
    all_results.append(r)
    if r['won']: wins += 1
    aligned_pct = r['aligned'] / r['steps'] * 100 if r['steps'] > 0 else 0
    print(f'  {g+1}/{N_EVAL}: {"WIN" if r["won"] else "fail"} ({r["steps"]}s, {aligned_pct:.0f}% aligned) {r["task_type"]} [{wins}/{g+1}={wins/(g+1)*100:.0f}%]')

print(f'\nResult: {wins}/{N_EVAL} = {wins/N_EVAL*100:.1f}%')
total_aligned = sum(r['aligned'] for r in all_results)
total_steps = sum(r['steps'] for r in all_results)
print(f'Overall alignment: {total_aligned}/{total_steps} = {total_aligned/total_steps*100:.0f}%')


In [ ]:
# Per task type breakdown
print('\n=== Per Task Type ===')
by_type = defaultdict(list)
for r in all_results:
    by_type[r['task_type']].append(r['won'])

for tt in sorted(by_type):
    outcomes = by_type[tt]
    w = sum(outcomes)
    print(f'  {tt}: {w}/{len(outcomes)} = {w/len(outcomes)*100:.0f}%')

# Per phase distribution
print('\n=== Phase Distribution ===')
phase_counts = Counter()
for r in all_results:
    phase_counts.update(r['phases'])
total_phases = sum(phase_counts.values())
for phase, count in phase_counts.most_common():
    print(f'  {phase}: {count} ({count/total_phases*100:.0f}%)')

# Action type distribution in wins vs fails
print('\n=== Action Types: Wins vs Fails ===')
win_actions = Counter()
fail_actions = Counter()
for r in all_results:
    c = win_actions if r['won'] else fail_actions
    c.update(r['action_types'])
print('  Wins:', dict(win_actions.most_common(5)))
print('  Fails:', dict(fail_actions.most_common(5)))


## 4. Error Analysis

Post-hoc study of failures: why did the agent fail?


In [ ]:
# === Error Analysis ===
failures = [r for r in all_results if not r['won']]
print(f'{len(failures)} failures to analyze')

# Failure patterns
print('\n=== Failure Patterns ===')

# 1. How many steps before failing?
step_counts = [r['steps'] for r in failures]
print(f'  Mean steps before failure: {np.mean(step_counts):.1f} (max={MAX_STEPS})')
print(f'  Timeout failures (hit MAX_STEPS): {sum(1 for s in step_counts if s >= MAX_STEPS)}')

# 2. What phase were they in when they failed?
print('\n=== Final Phase at Failure ===')
final_phases = Counter(r['phases'][-1] if r['phases'] else '?' for r in failures)
for phase, count in final_phases.most_common():
    print(f'  {phase}: {count}')

# 3. What task types fail most?
print('\n=== Failure Rate by Task Type ===')
for tt in sorted(by_type):
    outcomes = by_type[tt]
    fail_rate = 1 - sum(outcomes)/len(outcomes)
    print(f'  {tt}: {fail_rate*100:.0f}% failure rate ({len(outcomes)} games)')

# 4. Repetitive actions (agent stuck in loop)
print('\n=== Stuck in Loop? (same action 3+ times in a row) ===')
loop_count = 0
for r in failures:
    actions = [h['action'] for h in r['history']]
    for i in range(len(actions)-2):
        if actions[i] == actions[i+1] == actions[i+2]:
            loop_count += 1
            print(f'  Game "{r["task"][:30]}": repeated "{actions[i]}" at step {i+1}')
            break
print(f'  Total games with loops: {loop_count}/{len(failures)}')

# 5. Did retrieval help at all? Compare steps where candidates existed
print('\n=== Retrieval Quality ===')
# TODO: add retrieval metadata to step_log for deeper analysis
print('  (Enable inspection mode for per-step retrieval analysis)')


In [ ]:
# === Inspect worst failures ===
print('=== Worst Failures (full action history) ===')
# Sort failures by step count (shortest = worst, couldn\'t even start)
worst = sorted(failures, key=lambda r: r['steps'])[:3]

for r in worst:
    print(f'\nTask: {r["task"]}')
    print(f'Type: {r["task_type"]}, Steps: {r["steps"]}')
    for i, h in enumerate(r['history'][:10]):
        print(f'  {i+1}. {h["action"]} → {h["obs"][:60]}')
    if len(r['history']) > 10:
        print(f'  ... ({len(r["history"])-10} more steps)')
    print()


In [ ]:
# Save for comparison across runs
results_data = {
    'config': {'model': LLM_MODEL, 'n_eval': N_EVAL, 'top_k': TOP_K},
    'summary': {'wins': wins, 'total': N_EVAL, 'rate': wins/N_EVAL},
    'per_task': {tt: {'wins': sum(o), 'total': len(o)} for tt, o in by_type.items()},
    'games': [{'task': r['task'], 'task_type': r['task_type'], 'won': r['won'], 
               'steps': r['steps']} for r in all_results],
}

with open('../data/episodic/e9_pipeline_results.json', 'w') as f:
    json.dump(results_data, f, indent=2)
print(f'Saved to data/episodic/e9_pipeline_results.json')
